In [ ]:
from __future__ import division
import numpy as np 
import pickle
import scipy.io as sio
import scipy.sparse as ss 

from sklearn.preprocessing import binarize
from sklearn.svm import LinearSVC, NuSVC, SVC 
from sklearn.tree import DecisionTreeClassifier, ExtraTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier 
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.neural_network import MLPClassifier
from sklearn import metrics

np.random.seed(12345)

In [ ]:
# training data:
# train_feats = BOW linguistic features
# train_image_feats = BOC image categorical features
# train_y = FOILED or NOT FOILED => IF FOILED = 1, else 0
train_feats = sio.mmread('data_new/train_feats.mtx')
train_image_feats = sio.mmread('data_new/train_image_feats.mtx')
train_target = np.array(sio.mmread('data_new/train_y.mtx').todense()).tolist()[0]
#############
# validation data: Same pattern as training; 
# The testing data comes from the karpathy 5k validation set only. 
val_feats = sio.mmread('data_new/test_feats.mtx')
val_image_feats = sio.mmread('data_new/test_image_feats.mtx')
val_target = np.array(sio.mmread('data_new/test_y.mtx').todense()).tolist()[0]

In [ ]:
# Data processing, concatinating images with
# linguistic features and image features 
X_train = ss.hstack([(train_feats), train_image_feats])
#X_train = ss.hstack([binarize(train_feats), train_image_feats])
#X_train = train_feats

X_val = ss.hstack([(val_feats), val_image_feats])
#X_val = val_feats
#X_val = ss.hstack([binarize(val_feats), val_image_feats])

Y_train = np.array(train_target)
Y_test = np.array(val_target)

In [ ]:
# Logistic Regression with 'l1' penalty
logistic = LogisticRegression(penalty='l1')
logistic.fit(X_train, Y_train)
print 'Accuracy = ', metrics.accuracy_score(Y_test, logistic.predict(X_val))
target_names = ['REAL', 'FAKE']
print metrics.classification_report(Y_test, logistic.predict(X_val), 
                                    target_names=target_names)

In [ ]:
cmat = metrics.confusion_matrix(Y_test, logistic.predict(X_val))
print cmat 
print cmat.diagonal()/cmat.sum(axis=1)


#cmat.sum(axis=1)

In [ ]:
# Linear Support Vector Classifier with l2 regularizer and hinge loss
linearsvc = LinearSVC(penalty='l2', loss='hinge', verbose=1)
linearsvc.fit(X_train, Y_train)
print 'Accuracy = ', metrics.accuracy_score(Y_test, linearsvc.predict(X_val))
target_names = ['REAL', 'FAKE']
print metrics.classification_report(Y_test, linearsvc.predict(X_val), 
                                    target_names=target_names)

In [ ]:
# Decision tree classifier
decisiontree = DecisionTreeClassifier(random_state=0, class_weight='balanced')
decisiontree.fit(X_train, Y_train)
print 'Accuracy = ', metrics.accuracy_score(Y_test, decisiontree.predict(X_val))
target_names = ['REAL', 'FAKE']
print metrics.classification_report(Y_test, decisiontree.predict(X_val), 
                                    target_names=target_names)


In [ ]:
#print metrics.confusion_matrix(Y_test, decisiontree.predict(X_val))
help(DecisionTreeClassifier)

In [ ]:
# standard Gradient Boosting Classifier 
gb = GradientBoostingClassifier()
gb.fit(X_train, Y_train)
print 'Accuracy = ', metrics.accuracy_score(Y_test, gb.predict(X_val.toarray()))
target_names = ['REAL', 'FAKE']
print metrics.classification_report(Y_test, gb.predict(X_val.toarray()), 
                                    target_names=target_names)

In [ ]:
# extremely randomized tree classifier.
ert = ExtraTreeClassifier(splitter='best')
ert.fit(X_train, Y_train)
print 'Accuracy = ', metrics.accuracy_score(Y_test, ert.predict(X_val.toarray()))
target_names = ['REAL', 'FAKE']
print metrics.classification_report(Y_test, ert.predict(X_val), 
                                    target_names=target_names)

In [ ]:
pickle.dump(decisiontree, open('decisiontreeclassifiermodel.pkl', 'w'))

In [ ]:
# MLP Classifier as in the FOIL paper
mlp = MLPClassifier(solver='adam', alpha=1e-5, random_state=1)
mlp.fit(X_train, Y_train)
print 'Accuracy = ', metrics.accuracy_score(Y_test, mlp.predict(X_val.toarray()))
target_names = ['REAL', 'FAKE']
print metrics.classification_report(Y_test, mlp.predict(X_val), 
                                    target_names=target_names, digits=4)